# Klasyfikacja tekstów

Na dzisiejszych tekstach nauczymy się korzystać z poznanych ostatnio metod reprezentacji tekstów w zadaniu klasyfikacji. Będziemy pracować również na zbiorze danych poznanym (pobieżnie) w trakcie poprzednich zajęć. Tym razem będzie nas interesować relacja między samymi tekstami, a przypisanymi im etykietami oznaczającymi ładunek emocjonalny.

Wczytajmy dane, konkatenując (po kolumnach) informacje z dwu plików:

In [ ]:
import pandas
from pathlib import Path

DATA_DIR = Path("../../data")
EMO_DIR = DATA_DIR / "2024-emotion-recognition" / "train"
text_file = EMO_DIR / "in.tsv"
text_df = pandas.read_csv(text_file, sep="\t")
label_file = EMO_DIR / "expected.tsv"
label_df = pandas.read_csv(label_file, sep="\t")
emo_df = pandas.concat([text_df, label_df], axis=1)
emo_df

Spróbujmy spojrzeć na strukturę etykiet:

In [ ]:
emo_df[["Positive", "Negative"]].value_counts().sort_index()

Widzimy już pierwsze zaskakujące własności tego datasetu. Pracując z danymi zawsze warto zrobić sesję eksploracji danych i zweryfikować swoje aprioryczne założenia na temat ich struktury i własności. Cennym jest zapoznanie się z dokumentacją dostarczoną do danych co pozwoli odróżnić błędy i zanieczyszczenia w danych od celowych zabiegów twórców. Nasz dataset pochodzi z https://github.com/poleval/2024-emotion-recognition

Przygotujmy klasyczny podział danych na trzy części:
1. train - służy do treningu pojedynczego modelu, z określoną kombinacją hiperparametrów
2. dev - służy do selekcji najlepszej kombinacji hiperparametrów, model nie ma dostępu do tych danych w trakcie treningu
3. test - służy do ewaluacji jakości wybranego modelu, te dane nie powinny być podstawą selekcji modelu

In [ ]:
from sklearn.model_selection import train_test_split

SEED = 42

train_df, non_train_df = train_test_split(emo_df, test_size=0.2, random_state=SEED)
dev_df, test_df = train_test_split(non_train_df, test_size=0.5, random_state=SEED)

Spróbujmy przygotować reprezentację wektorową tekstów zaczynając od metody TF-IDF, wektoryzator trenujemy na podstawie danych treningowych, tak żeby informacje leksykalne również nie przedostały się do finalnego modelu:

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer()
tfidf_train_vectors = tfidf.fit_transform(train_df["text"])

Najpierw zajmiemy się treningiem klasyfikatora binarnego, i.e. będzie nas interesowała pojedyncza etykieta, np. `Positive`.

Skorzystamy z regresji logistycznej, która każdej z cech przypisuje współczynnik istotności, po czym ważona współczynnikami suma cech przekształcona logarytmicznie jest mapowana na prawdopodobieństwo:

In [ ]:
import numpy
from sklearn.linear_model import LogisticRegression

target_label = "Positive"

train_labels = numpy.array(train_df[target_label])
log_reg = LogisticRegression(random_state=SEED)
_ = log_reg.fit(tfidf_train_vectors, train_labels)

Ogromną zaletą regresji logistycznej zastosowanej do reprezentacji leksykalnych jest prostota i interpretowalność: niektóre słowa "pchają" decyzję w stronę pozytywną, inne zaś w stronę negatywną:

In [ ]:
coefficients = log_reg.coef_.argsort()
non_pos = coefficients[0,:10]
pos = coefficients[0,-10:]
vocab = tfidf.get_feature_names_out()
non_pos_words = [vocab[i] for i in non_pos]
pos_words = [vocab[i] for i in pos]
print(f"Non-{target_label} features: {non_pos_words}")
print(f"{target_label} features: {pos_words}")

Zobaczmy jak sprawuje się model na dev secie:

In [ ]:
tfidf_dev_vectors = tfidf.transform(dev_df["text"])
dev_labels = numpy.array(dev_df[target_label])
dev_preds = log_reg.predict(tfidf_dev_vectors)
accuracy = (dev_preds == dev_labels).mean()
print(f"Accuracy: {accuracy * 100:.2f}%")

Przeważnie w naszej ewaluacji chcemy jednak skorzystać z bardziej wyrafinowanych metryk. Klasyfikacja binarna nie jest symetryczna, fałszywy wynik pozytywny może okazać się dużo bardziej kosztowny niż fałszywy negatywny, i vice versa. Dlatego przeważnie korzysta się w takim zadaniu z:
1. `precision` - proporocji negatywnych obiektów, rozpoznanych jako negatywne (swoistość)
2. `recall` - proporcji pozytywnych obiektów, rozpoznanych jako pozytywne (czułość)
3. `f1` - średniej harmonicznej `precision` i `recall`

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

p, r, f1, _ = precision_recall_fscore_support(dev_labels, dev_preds, average="binary")
for lab, score in zip(["precision", "recall", "f1"], [p,r,f1]):
    print(f"{lab}: {round(score * 100, 2)}%")

W tym wypadku klasyfikator jest zbyt "liberalny" w przyznawaniu pozytywnych etykiet.

Regresja logistyczna dla każdej obserwacji przypisuje liczbową "siłę przekonania" - *confidence* względem etykiety pozytywnej, jeżeli jest ona wyższa od $0.5$, interpretujemy to jako wartość pozytywną. Zależność poprawności modelu od wartości *confidence* nazywamy kalibracją. Dobrze skalibrowany model to taki, którego pewność liniowo koreluje z proporcją poprawnie przyznanych etykiet, przeważnie przedstawia się to na wykresie, na którym dla decyzji pozytywnych klasyfikatora przedstawiamy jaka proporcja była prawidłowa w zależności od pewności modelu:

In [ ]:
from sklearn.calibration import calibration_curve
from matplotlib import pyplot as plt

NUM_BINS = 10
dev_probs = log_reg.predict_proba(tfidf_dev_vectors).T[1]
prob_true, prob_pred = calibration_curve(dev_labels, dev_probs, n_bins=NUM_BINS, strategy="uniform")
plt.bar(prob_pred, prob_true, width=1/NUM_BINS)
plt.plot((0,1), (0,1), c="red", linestyle="--")
plt.ylim(0,1)
plt.xlim(0, 1)
plt.ylabel("True frequency")
plt.xlabel("Confidence")
plt.show()

Poznaliśmy liczbowe wskaźniki jakości modelu, dalej jednak nie wiemy, czy świadczy to o tym, że nasz model jest dobry, czy marny. Na to ostatnie pytanie możemy sobie odpowiedzieć dopiero posiadając jakiś punkt referencyjny. Przygotujmy dwa takie *baseline*'y:

In [ ]:
import numpy
numpy.random.seed(SEED)

random_preds = numpy.random.random(dev_preds.shape) >= 0.5
most_common_label = train_df[target_label].astype(int).mode()[0]
freq_preds = numpy.ones(dev_preds.shape) * most_common_label
clf_eval = []
for clf_name, preds in [
    ("random", random_preds),
    ("freq", freq_preds),
    ("LR", dev_preds),
]:
    p, r, f1, _ = precision_recall_fscore_support(dev_labels, preds, average="binary")
    clf_eval.append({"name": clf_name, "p":p, "r":r, "f1": f1})

pandas.DataFrame(clf_eval).set_index("name")

Widzimy, że klasyfikator nie jest wcale o wiele lepszy od najprostszych technik generowania predykcji. To one wyznaczają naturalną "podłogę"
naszych oczekiwań. W wypadku realistycznych zbiorów danych sufit także istnieje, i jest wyznaczony przez jakość datasetu, to ile błędów może zawierać. Możemy czasem szacować taką liczbę podając wartości określające wewnętrzną spójność procesu anotacji.

## Ćwiczenie 1:

Dla każdej wartości w zakresie $0, 0.01, 0.02 ... 0.99, 1$ oblicz jakie predykcje wydałby klasyfikator. Następnie sprawdź jaka jest proporcja wyników fałszywie pozytywnych, i prawdziwie pozytywnych. Zbierz te dane, i przedstaw na wykresie jaka jest między nimi zależność, i.e. jak wzrost proporcji wynikow fałszywie pozytywnych idzie w parze z wzrostem proporcji prawdziwie pozytywnych. Przedstaw tę zależnosc na wykresie

In [ ]:
# TODO

Chcemy jakoś usprawnić nasz klasyfikator. Na razie pozostawmy architekturę bez zmian, i zastanówmy się nad tym, ile informacji wyciągamy z samych danych. Zwróćmy uwagę, że reprezentacja TF-IDF traci informację o wielkości liter. To na pewno jest cecha która wiąże się z emocjami autorów:

In [ ]:
train_is_allcaps = train_df["text"].apply(lambda x: x == x.upper())
print("allcaps share:", train_is_allcaps.value_counts())

Aby nasz klasyfikator mógł skorzystać ze wskazanej cechy, wystarczy dokonkatenować ją (jako wektor) do istniejącej reprezentacji macierzowej danych. Napiszmy w tym celu funkcję pomocniczą:

In [ ]:
import scipy

def join_feat(vectors, feat):
    sparse_feat = scipy.sparse.csr_array([[val] for val in feat.tolist()])
    return scipy.sparse.hstack([vectors, sparse_feat])

aug_train_vectors = join_feat(tfidf_train_vectors, train_is_allcaps)
dev_is_allcaps = dev_df["text"].apply(lambda x: x == x.upper())
aug_dev_vectors = join_feat(tfidf_dev_vectors, dev_is_allcaps)
aug_train_vectors

Wytrenujmy klasyfikator i sprawdźmy wyniki:

In [ ]:
aug_log_reg = LogisticRegression(random_state=SEED)
_ = aug_log_reg.fit(aug_train_vectors, train_labels)
aug_preds = aug_log_reg.predict(aug_dev_vectors)
p, r, f1, _ = precision_recall_fscore_support(dev_labels, aug_preds, average="binary")
clf_eval.append({"name": "aug", "p":p, "r":r, "f1": f1})
pandas.DataFrame(clf_eval).set_index("name")

## Ćwiczenie 2:

Spróbuj zaproponować inne cechy, które mogą pozytywnie wpłynąć na wyniki klasyfikatora. Przeprowadź eksperyment sprawdzający Twoją hipotezę

In [ ]:
# TODO

Spróbujmy zareprezentować nasze dane korzystając z wektoryzacji gęstej, używanymi ostatnio embeddingami:

In [ ]:
import re

DIM = 100

tokenizer = re.compile(r"\b\w+\b")
w2v_path = DATA_DIR / "others" / "wiki-forms-all-100-skipg-ns-30-it100.txt"
w2v = pandas.read_csv(w2v_path, sep=" ", skiprows=[0], header=None)
emb_arr = numpy.vstack(w2v.set_index(0).apply(numpy.array, axis=1))
ind2w = w2v[0].to_dict()
w2ind = {w: ind for ind, w in ind2w.items()}

def get_vec(word):
    ind = w2ind.get(word, None)
    if ind:
        return emb_arr[ind]
    return numpy.zeros((DIM))

def text_to_vec(text):
    tokens = tokenizer.findall(text)
    if not tokens:
        return numpy.zeros(DIM)
    vecs = [get_vec(tok.lower()) for tok in tokens]
    arr = numpy.vstack(vecs)
    mean_vec = arr.mean(axis=0)
    return mean_vec

dense_train_vectors = numpy.vstack([text_to_vec(text) for text in train_df["text"]])
dense_dev_vectors = numpy.vstack([text_to_vec(text) for text in dev_df["text"]])

Sprawdźmy jak zachowa się taki klasyfikator:

In [ ]:
dense_log_reg = LogisticRegression(random_state=SEED)
_ = dense_log_reg.fit(dense_train_vectors, train_labels)
dense_preds = dense_log_reg.predict(dense_dev_vectors)
p, r, f1, _ = precision_recall_fscore_support(dev_labels, dense_preds, average="binary")
clf_eval.append({"name": "dense", "p":p, "r":r, "f1": f1})
pandas.DataFrame(clf_eval).set_index("name")

## Ćwiczenie 3:

Wytrenuj klasyfikator hybrydowy, tj. taki, który będzie miał dostęp zarówno do reprezentacji gęstej, jak i rzadkiej badanych tekstów. Przeprowadź jego ewaluację zgodnie z powyższą metodą. sprawdź jakie współczynniki regresja przypisze cechom gęstym

In [ ]:
# TODO

Do tej pory pracowaliśmy z klasyfikacją binarną. Spróbujmy teraz zobaczyć, z jakimi wyzwaniami wiąże się klasyfikacja z więcej niż dwoma etykietami. Widzieliśmy wcześniej, że dane nie traktują etykiet "Positive", "Neutral", i " Negative" jako wykluczających się. Wprowadźmy jednak takie założenie, i wyfiltrujmy dane, odrzucając przypadki gdzie nie jest ono spełnione:

In [ ]:
multiclasses = ["Negative", "Neutral", "Positive"]
nonambivalent = emo_df[multiclasses].sum(axis=1) == 1
multi_df = emo_df[nonambivalent]
print(f"{len(multi_df)}/{len(emo_df)} examples saved")
multi_train_df, non_multi_train_df = train_test_split(multi_df, test_size=0.2, random_state=SEED)
multi_dev_df, multi_test_df = train_test_split(non_multi_train_df, test_size=0.5, random_state=SEED)

Zakodujmy dane. W tym celu musimy wpierw zakodować etykiety klas - każdej z nich przypiszemy jedną liczbę naturalną (od 0 do 2) zgodnie z kolejnością zadaną w liście `multiclasses`. Do reprezentacji tekstu wykorzystajmy embeddingi gęste:

In [ ]:
multi_train_labels = multi_train_df[multiclasses].apply(lambda x: x.tolist().index(True), axis=1)
multi_dev_labels = multi_dev_df[multiclasses].apply(lambda x: x.tolist().index(True), axis=1)
multi_dense_train_vectors = numpy.vstack([text_to_vec(text) for text in multi_train_df["text"]])
multi_dense_dev_vectors = numpy.vstack([text_to_vec(text) for text in multi_dev_df["text"]])

Wytrenujmy klasyfikator, i spójrzmy na wyuczone współczynniki:

In [ ]:
multi_dense_log_reg = LogisticRegression(random_state=SEED)
_ = multi_dense_log_reg.fit(multi_dense_train_vectors, multi_train_labels)
multi_dense_log_reg.coef_

Zobaczmy jak efektywny jest nasz klasyfikator. W tym celu zastosujemy te same metryki, będą jednak one liczone stosując podejście "one-vs-rest" - metryka M dla klasy X, oznacza liczenie standardowej metryki M w wersji binarnej na uproszczonym problemie w którym etykiety klas innych niż X konwertujemy do wartości negatywnej:

In [ ]:
multi_dense_preds = multi_dense_log_reg.predict(multi_dense_dev_vectors)
p, r, f1, s = precision_recall_fscore_support(multi_dev_labels, multi_dense_preds)
multi_eval = pandas.DataFrame({"precision": p, "recall": r, "f1": f1, "support": s}, index=multiclasses)
multi_eval.loc["macro_avg"] = multi_eval[["precision", "recall", "f1"]].mean(axis=0)
multi_eval

## Ćwiczenie 4:

Oblicz wartości *confidence* dla każdego z przykładów z dev-setu względem każdej z klas. Wykorzystaj entropię, żeby jedną liczbą scharakteryzować niepewność modelu. Następnie policz te same wartości dla komplementarnego podzbioru danych ambiwalentnych (i.e. takich, które łamią założenie o wykluczaniu użytych etykiet)

Wzór entropii: $-\sum_{i=1}^{n} p(x_{i}) log(p(x_{i}))$

In [ ]:
# TODO